In [ ]:
%py
%pip install pytest

spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for comprehensive testing of the refactored Sales KPI pipeline classes and functions
# Purpose: Validate correctness, schema, data types, error handling, and business logic of the refactored pipeline
# Author: Giang Nguyen
# Date: 2025-10-13
# Description: This script tests the refactored PySpark pipeline for generating sales KPIs, including schema validation, data type conversion, NULL handling, business logic, Delta Lake operations, and error scenarios. It uses Unity Catalog tables as sources and validates the output table against expected results.

# --- Required imports for PySpark testing ---
from pyspark.sql import Row  
from pyspark.sql import DataFrame  
from pyspark.sql import functions as F  
from pyspark.sql.types import (  
    StructType, StructField, IntegerType, StringType, DoubleType, LongType, DateType, TimestampType
)
from pyspark.sql.utils import AnalysisException  
from pyspark.sql.window import Window  
import pytest  

# --- Test utility functions and classes ---

class SalesKPIDataLoader:
    """
    Class to load and validate input data from Unity Catalog tables.

    Args:
        spark (SparkSession): Spark session object.

    Methods:
        load_table(table_name: str, schema: StructType) -> DataFrame:
            Loads a table and validates its schema.
    """
    def __init__(self, spark):
        self.spark = spark

    def load_table(self, table_name: str, schema: StructType) -> DataFrame:
        """
        Loads a table from Unity Catalog and validates its schema.

        Args:
            table_name (str): Fully qualified table name.
            schema (StructType): Expected schema.

        Returns:
            DataFrame: Loaded DataFrame with validated schema.
        """
        try:
            df = self.spark.read.table(table_name)
        except AnalysisException as e:
            raise RuntimeError(f"Input table {table_name} not found") from e
        # Schema validation: number of columns and types
        actual_fields = df.schema.fields
        expected_fields = schema.fields
        assert len(actual_fields) == len(expected_fields), f"Column count mismatch in {table_name}"
        for af, ef in zip(actual_fields, expected_fields):
            assert af.name == ef.name, f"Column name mismatch: {af.name} != {ef.name} in {table_name}"
            assert isinstance(af.dataType, type(ef.dataType)), f"Type mismatch for {af.name} in {table_name}"
        return df

class SalesKPIPipeline:
    """
    Class encapsulating the sales KPI pipeline logic.

    Args:
        spark (SparkSession): Spark session object.

    Methods:
        run_pipeline() -> DataFrame:
            Executes the pipeline and returns the final KPI DataFrame.
    """
    def __init__(self, spark):
        self.spark = spark
        self.loader = SalesKPIDataLoader(spark)

    def run_pipeline(self) -> DataFrame:
        """
        Executes the sales KPI pipeline.

        Returns:
            DataFrame: Final KPI DataFrame.
        """
        # Define expected schemas for input tables
        product_schema = StructType([
            StructField("product_id", StringType(), True),
            StructField("product_name", StringType(), True),
            StructField("market_segment", StringType(), True)
        ])
        sales_schema = StructType([
            StructField("transaction_id", StringType(), True),
            StructField("sales_product_id", StringType(), True),
            StructField("sales_amount", LongType(), True),
            StructField("sales_date", DateType(), True)
        ])
        marketshare_schema = StructType([
            StructField("ms_product_id", StringType(), True),
            StructField("market_share_pct", DoubleType(), True)
        ])
        # Load tables with schema validation
        product_df = self.loader.load_table("purgo_playground.product_data", product_schema)
        sales_df = self.loader.load_table("purgo_playground.product_sales_data", sales_schema)
        market_share_df = self.loader.load_table("purgo_playground.product_marketshare_data", marketshare_schema)
        # Data cleaning and renaming
        sales_df = sales_df.filter(F.col("sales_amount").isNotNull()) \
                           .withColumn("sales_year", F.year(F.col("sales_date")))
        # Join product info
        sales_enriched_df = sales_df.join(product_df, sales_df.sales_product_id == product_df.product_id, "left") \
                                    .drop(product_df.product_id)
        # Join market share info
        sales_market_df = sales_enriched_df.join(market_share_df, sales_enriched_df.sales_product_id == market_share_df.ms_product_id, "left") \
                                           .drop("ms_product_id")
        # KPI calculation
        window_spec = Window.partitionBy("sales_product_id").orderBy("sales_year")
        sales_agg_df = sales_market_df.groupBy("sales_product_id", "sales_year", "product_name", "market_segment") \
            .agg(
                F.sum("sales_amount").alias("total_sales"),
                F.round(F.avg("market_share_pct"), 2).alias("avg_market_share")
            )
        sales_agg_df = sales_agg_df.withColumn("prev_year_sales", F.lag("total_sales", 1).over(window_spec))
        sales_agg_df = sales_agg_df.withColumn("yoy_growth_pct",
            F.round(((F.col("total_sales") - F.col("prev_year_sales")) / F.col("prev_year_sales")) * 100, 2)
        )
        sales_agg_df = sales_agg_df.withColumn("market_penetration_flag",
            F.when(F.col("avg_market_share") > 25, F.lit("High"))
             .when((F.col("avg_market_share") <= 25) & (F.col("avg_market_share") >= 10), F.lit("Medium"))
             .otherwise(F.lit("Low"))
        )
        # Sales rank
        rank_window = Window.partitionBy("sales_year").orderBy(F.col("total_sales").desc())
        sales_agg_df = sales_agg_df.withColumn("sales_rank", F.row_number().over(rank_window))
        # Final output
        final_df = sales_agg_df.select(
            "sales_year", "sales_product_id", "product_name", "market_segment",
            "total_sales", "prev_year_sales", "yoy_growth_pct",
            "avg_market_share", "market_penetration_flag", "sales_rank"
        )
        return final_df

    def write_output(self, df: DataFrame, table_name: str):
        """
        Writes the final DataFrame to a managed table in overwrite mode.

        Args:
            df (DataFrame): DataFrame to write.
            table_name (str): Fully qualified output table name.

        Returns:
            None
        """
        try:
            df.write.mode("overwrite").saveAsTable(table_name)
        except Exception as e:
            raise RuntimeError(f"Failed to write to {table_name}") from e

# --- Test functions ---

def test_schema_validation(spark):
    """
    Test that input tables have correct schema and column count.
    """
    loader = SalesKPIDataLoader(spark)
    # Define expected schemas
    product_schema = StructType([
        StructField("product_id", StringType(), True),
        StructField("product_name", StringType(), True),
        StructField("market_segment", StringType(), True)
    ])
    sales_schema = StructType([
        StructField("transaction_id", StringType(), True),
        StructField("sales_product_id", StringType(), True),
        StructField("sales_amount", LongType(), True),
        StructField("sales_date", DateType(), True)
    ])
    marketshare_schema = StructType([
        StructField("ms_product_id", StringType(), True),
        StructField("market_share_pct", DoubleType(), True)
    ])
    # Load and validate schemas
    for table, schema in [
        ("purgo_playground.product_data", product_schema),
        ("purgo_playground.product_sales_data", sales_schema),
        ("purgo_playground.product_marketshare_data", marketshare_schema)
    ]:
        df = loader.load_table(table, schema)
        assert df.schema == schema, f"Schema mismatch for {table}"

def test_data_type_conversion(spark):
    """
    Test that data type conversions (e.g., market_share_pct to double) work as expected.
    """
    loader = SalesKPIDataLoader(spark)
    marketshare_schema = StructType([
        StructField("ms_product_id", StringType(), True),
        StructField("market_share_pct", DoubleType(), True)
    ])
    df = loader.load_table("purgo_playground.product_marketshare_data", marketshare_schema)
    # Check that market_share_pct is DoubleType and NULLs are handled
    assert isinstance(df.schema["market_share_pct"].dataType, DoubleType)
    null_count = df.filter(F.col("market_share_pct").isNull()).count()
    assert null_count >= 0  # Should not error

def test_null_handling(spark):
    """
    Test that rows with NULLs in required fields are excluded from KPI calculation.
    """
    pipeline = SalesKPIPipeline(spark)
    final_df = pipeline.run_pipeline()
    # sales_amount is required, so no NULLs in total_sales
    assert final_df.filter(F.col("total_sales").isNull()).count() == 0
    # prev_year_sales and yoy_growth_pct can be NULL
    null_prev_year = final_df.filter(F.col("prev_year_sales").isNull()).count()
    null_yoy_growth = final_df.filter(F.col("yoy_growth_pct").isNull()).count()
    assert null_prev_year >= 0
    assert null_yoy_growth >= 0

def test_kpi_business_logic(spark):
    """
    Test business logic for YoY Growth, Market Penetration Flag, and Sales Rank.
    """
    pipeline = SalesKPIPipeline(spark)
    final_df = pipeline.run_pipeline()
    # Market Penetration Flag logic
    high_flag = final_df.filter(F.col("avg_market_share") > 25).select("market_penetration_flag").distinct().collect()
    assert any(row.market_penetration_flag == "High" for row in high_flag)
    medium_flag = final_df.filter((F.col("avg_market_share") <= 25) & (F.col("avg_market_share") >= 10)).select("market_penetration_flag").distinct().collect()
    assert any(row.market_penetration_flag == "Medium" for row in medium_flag)
    low_flag = final_df.filter((F.col("avg_market_share") < 10) | (F.col("avg_market_share").isNull())).select("market_penetration_flag").distinct().collect()
    assert any(row.market_penetration_flag == "Low" for row in low_flag)
    # YoY Growth calculation
    sample = final_df.filter(F.col("sales_product_id") == "PROD001").orderBy("sales_year").collect()
    if len(sample) >= 2:
        prev, curr = sample[0], sample[1]
        expected_growth = round(((curr.total_sales - prev.total_sales) / prev.total_sales) * 100, 2) if prev.total_sales else None
        assert curr.yoy_growth_pct == expected_growth
    # Sales Rank assignment
    for year in final_df.select("sales_year").distinct().collect():
        year_val = year.sales_year
        ranks = final_df.filter(F.col("sales_year") == year_val).select("total_sales", "sales_rank").orderBy(F.col("total_sales").desc()).collect()
        assert ranks[0].sales_rank == 1

def test_integration_end_to_end(spark):
    """
    Integration test: run full pipeline and validate output table.
    """
    pipeline = SalesKPIPipeline(spark)
    final_df = pipeline.run_pipeline()
    # Write output and read back
    pipeline.write_output(final_df, "purgo_playground.sales_kpi")
    output_df = spark.read.table("purgo_playground.sales_kpi")
    # Validate output schema
    expected_schema = StructType([
        StructField("sales_year", IntegerType(), True),
        StructField("sales_product_id", StringType(), True),
        StructField("product_name", StringType(), True),
        StructField("market_segment", StringType(), True),
        StructField("total_sales", LongType(), True),
        StructField("prev_year_sales", LongType(), True),
        StructField("yoy_growth_pct", DoubleType(), True),
        StructField("avg_market_share", DoubleType(), True),
        StructField("market_penetration_flag", StringType(), True),
        StructField("sales_rank", IntegerType(), True)
    ])
    assert output_df.schema == expected_schema
    # Validate data quality: no duplicate (sales_product_id, sales_year)
    group_count = output_df.groupBy("sales_product_id", "sales_year").count().filter(F.col("count") > 1).count()
    assert group_count == 0

def test_delta_lake_operations(spark):
    """
    Test Delta Lake operations: MERGE, UPDATE, DELETE on output table.
    """
    # MERGE: upsert a new row
    from delta.tables import DeltaTable  
    output_table = DeltaTable.forName(spark, "purgo_playground.sales_kpi")
    # Prepare upsert row
    upsert_row = Row(
        sales_year=2024, sales_product_id="PROD999", product_name="Test Product",
        market_segment="Test", total_sales=1000, prev_year_sales=None,
        yoy_growth_pct=None, avg_market_share=15.0, market_penetration_flag="Medium", sales_rank=99
    )
    upsert_df = spark.createDataFrame([upsert_row])
    # MERGE
    output_table.alias("tgt").merge(
        upsert_df.alias("src"),
        "tgt.sales_year = src.sales_year AND tgt.sales_product_id = src.sales_product_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    # Validate upsert
    result = spark.read.table("purgo_playground.sales_kpi").filter(
        (F.col("sales_year") == 2024) & (F.col("sales_product_id") == "PROD999")
    ).count()
    assert result == 1
    # UPDATE
    output_table.update(
        condition="sales_product_id = 'PROD999'",
        set={"product_name": "'Updated Product'"}
    )
    updated = spark.read.table("purgo_playground.sales_kpi").filter(
        (F.col("sales_year") == 2024) & (F.col("product_name") == "Updated Product")
    ).count()
    assert updated == 1
    # DELETE
    output_table.delete("sales_product_id = 'PROD999'")
    deleted = spark.read.table("purgo_playground.sales_kpi").filter(
        (F.col("sales_year") == 2024) & (F.col("sales_product_id") == "PROD999")
    ).count()
    assert deleted == 0

def test_error_missing_input_table(spark):
    """
    Test error handling for missing input table.
    """
    loader = SalesKPIDataLoader(spark)
    fake_schema = StructType([StructField("id", StringType(), True)])
    try:
        loader.load_table("purgo_playground.nonexistent_table", fake_schema)
        assert False, "Should have raised RuntimeError for missing table"
    except RuntimeError as e:
        assert "not found" in str(e)

def test_error_output_table_write_failure(spark, monkeypatch):
    """
    Test error handling for output table write failure.
    """
    pipeline = SalesKPIPipeline(spark)
    final_df = pipeline.run_pipeline()
    def fail_write(*args, **kwargs):
        raise Exception("Simulated write failure")
    monkeypatch.setattr(final_df.write, "saveAsTable", fail_write)
    try:
        pipeline.write_output(final_df, "purgo_playground.sales_kpi")
        assert False, "Should have raised RuntimeError for write failure"
    except RuntimeError as e:
        assert "Failed to write" in str(e)

def test_performance_large_data(spark):
    """
    Performance test: run pipeline on large synthetic data and measure execution time.
    """
    import time  
    pipeline = SalesKPIPipeline(spark)
    start = time.time()
    final_df = pipeline.run_pipeline()
    duration = time.time() - start
    # Assert pipeline completes within reasonable time (e.g., < 60s for test data)
    assert duration < 60

def test_complex_type_handling(spark):
    """
    Test that complex types (ARRAY, STRUCT, MAP) are not present in output schema.
    """
    output_df = spark.read.table("purgo_playground.sales_kpi")
    for field in output_df.schema.fields:
        assert not (field.dataType.typeName() in ["array", "struct", "map"]), f"Complex type found in {field.name}"

def test_column_count_matches_schema(spark):
    """
    Test that number of columns in output matches target table schema.
    """
    output_df = spark.read.table("purgo_playground.sales_kpi")
    expected_columns = [
        "sales_year", "sales_product_id", "product_name", "market_segment",
        "total_sales", "prev_year_sales", "yoy_growth_pct",
        "avg_market_share", "market_penetration_flag", "sales_rank"
    ]
    assert output_df.columns == expected_columns

def test_window_function_analytics(spark):
    """
    Test window function for sales rank and YoY growth.
    """
    output_df = spark.read.table("purgo_playground.sales_kpi")
    # Validate sales_rank is sequential per year
    for year in output_df.select("sales_year").distinct().collect():
        year_val = year.sales_year
        ranks = output_df.filter(F.col("sales_year") == year_val).select("sales_rank").orderBy("sales_rank").collect()
        assert ranks[0].sales_rank == 1

def test_data_quality_validation(spark):
    """
    Test data quality: no NULLs in required output fields, valid flag values.
    """
    output_df = spark.read.table("purgo_playground.sales_kpi")
    # Required fields: sales_year, sales_product_id, total_sales
    assert output_df.filter(F.col("sales_year").isNull()).count() == 0
    assert output_df.filter(F.col("sales_product_id").isNull()).count() == 0
    assert output_df.filter(F.col("total_sales").isNull()).count() == 0
    # Flag values
    valid_flags = {"High", "Medium", "Low"}
    flags = set(row.market_penetration_flag for row in output_df.select("market_penetration_flag").distinct().collect())
    assert flags.issubset(valid_flags)

# --- Main test runner ---
def run_all_tests(spark):
    """
    Runs all test functions for the Sales KPI pipeline.

    Args:
        spark (SparkSession): Spark session object.

    Returns:
        None
    """
    test_schema_validation(spark)
    test_data_type_conversion(spark)
    test_null_handling(spark)
    test_kpi_business_logic(spark)
    test_integration_end_to_end(spark)
    test_delta_lake_operations(spark)
    test_error_missing_input_table(spark)
    # test_error_output_table_write_failure(spark, monkeypatch)  # monkeypatch only in pytest context
    test_performance_large_data(spark)
    test_complex_type_handling(spark)
    test_column_count_matches_schema(spark)
    test_window_function_analytics(spark)
    test_data_quality_validation(spark)
    print("All Sales KPI pipeline tests passed.")

# Uncomment below to run all tests in Databricks notebook context
# run_all_tests(spark)
